In [ ]:
#BLOCK-1
#---------------------------------------------------------------------------

import torch
import os
import sys
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
from matplotlib import pyplot as plt

PROJECT_ROOT = "/content/Xray-temp-master/Xray-temp-master"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from models.full_model import DenseNetCBAM

print("Environment is ready.")

In [ ]:
#BLOCK-2
#---------------------------------------------------------------------------

# config
MODELS_TO_TEST = {
    "Model 1 (BCE Baseline)": "/content/drive/MyDrive/best_model_scientific(1).pth",
    "Model 2 (ASL)": "/content/drive/MyDrive/best_model_asl(1).pth",
    "Model 3 (Processed ASL)": "/content/drive/MyDrive/best_model_processed_asl.pth",
    "Model 4 (Final Best Model)": "/content/drive/MyDrive/absolute_best_model.pth"
}

IMG_DIR = "/content/dataset/FinalDataset_PA"
TEST_CSV = "/content/test_split.csv"
CLASS_LIST = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# random img to test
df_test = pd.read_csv(TEST_CSV)
df_has_disease = df_test[df_test[CLASS_LIST].sum(axis=1) > 0]
random_row = df_has_disease.sample(1).iloc[0]
img_basename = random_row['Image Index']
TEST_IMG = os.path.join(IMG_DIR, img_basename)
true_labels = [cls for cls in CLASS_LIST if random_row[cls] == 1]

# image
if os.path.exists(TEST_IMG):
    img_pil = Image.open(TEST_IMG).convert('RGB')
    plt.figure(figsize=(6, 6))
    plt.imshow(img_pil, cmap='gray')
    plt.title(f"Image Index: {img_basename}\nGround Truth: {', '.join(true_labels)}")
    plt.axis('off')
    plt.show()

In [ ]:
#BLOCK-3
#---------------------------------------------------------------------------

def predict_with_thresholds(image_path, model, device, class_names, model_name, true_labels_list, thresholds=None):
    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.sigmoid(output).cpu().numpy()[0]

    print(f"\n{model_name}")
    print("-" * 75)

    current_thresholds = thresholds if thresholds is not None else [0.5] * len(class_names)
    findings = []
    
    for i in range(len(class_names)):
        name = class_names[i]
        prob = probs[i]
        thresh = current_thresholds[i]
        is_detected = prob > thresh
        is_truth = name in true_labels_list

        findings.append({
            'name': name, 'prob': prob, 'thresh': thresh,
            'detected': is_detected, 'truth': is_truth
        })

    sorted_findings = sorted(findings, key=lambda x: x['prob'], reverse=True)

    print(f"| {'Pathology':<20} | {'Score':<8} | {'Thresh':<7} | {'Status':<10} | {'Note':<15} |")
    print(f"|{'-'*22}|{'-'*10}|{'-'*9}|{'-'*12}|{'-'*17}|")
    for f in sorted_findings[:5]:
        status = "POSITIVE" if f['detected'] else "Negative"
        truth_mark = "Ground Truth" if f['truth'] else ""
        print(f"| {f['name']:<20} | %{f['prob']*100:>6.2f} | {f['thresh']:.2f}  | {status:<10} | {truth_mark:<15} |")

In [ ]:
#BLOCK-4
#---------------------------------------------------------------------------

print(f"FILE: {img_basename}")
print(f"ACTUAL DIAGNOSIS: {', '.join(true_labels)}")

for model_name, model_path in MODELS_TO_TEST.items():
    if not os.path.exists(model_path):
        continue

    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model = DenseNetCBAM()

    num_classes = len(CLASS_LIST)
    # handle different layer naming conventions
    if hasattr(model, 'classifier'):
        model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
    elif hasattr(model, 'fc'):
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

    model.load_state_dict(checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint)
    model.to(device)

    # dynamic tresholds of model 4
    model_thresholds = checkpoint.get('thresholds', None) if "Model 4" in model_name else None

    predict_with_thresholds(TEST_IMG, model, device, CLASS_LIST, model_name, true_labels, model_thresholds)